### Data Transformation (like Feature Engineering)

In [1]:
1+1

2

In [2]:
import os
%pwd


'f:\\MS\\text-summarization-transformers\\research'

In [4]:
os.chdir("../")


In [5]:
%pwd

'f:\\MS\\text-summarization-transformers'

In [13]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: str

In [14]:
from src.text_summarization_transformers.constants import *
from src.text_summarization_transformers.utils.common import read_yaml, create_directories

In [15]:
class ConfiguartionManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir=Path(config.root_dir),
            data_path=Path(config.data_path),
            tokenizer_name=config.tokenizer_name
        )

        return data_transformation_config

In [16]:
import os
from src.text_summarization_transformers.logging import logger
from transformers import AutoTokenizer
from datasets import load_from_disk

### Data Transformation Component

In [19]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_name)

    def convert_examples_to_features(self, example_batch):
        input_encodings = self.tokenizer(example_batch['dialogue'], max_length=1024, truncation=True)

        target_encodings = self.tokenizer(text_target=example_batch['summary'], max_length=128, truncation=True)

        return {
            'input_ids': input_encodings['input_ids'],
            'attention_mask': input_encodings['attention_mask'],
            'labels': target_encodings['input_ids']
    }

    def convert(self):
        dataset_samsum = load_from_disk(self.config.data_path)
        dataset_samsum_pt = dataset_samsum.map(self.convert_examples_to_features, batched = True)
        dataset_samsum_pt.save_to_disk(os.path.join(self.config.root_dir,"samsum_dataset"))

In [20]:
config = ConfiguartionManager()
data_transformation_config = config.get_data_transformation_config()
data_transformation = DataTransformation(config=data_transformation_config)
data_transformation.convert()

[2026-05-29 02:20:27,161:INFO:yaml file: config\config.yaml loaded successfully]
[2026-05-29 02:20:27,166:INFO:yaml file: params.yaml loaded successfully]
[2026-05-29 02:20:27,169:INFO:created directory at: artifacts]
[2026-05-29 02:20:27,169:INFO:created directory at: artifacts/data_transformation]
[2026-05-29 02:20:27,413:INFO:HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-05-29 02:20:27,445:INFO:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"]
[2026-05-29 02:20:27,486:INFO:HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-05-29 02:20:27,517:INFO:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/to

Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 121093.45 examples/s]
